In [1]:
# Cell 1 — Imports
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
import pickle
import math
from torch.utils.data import Dataset, DataLoader

In [2]:
# Cell 2 — Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [3]:
# Cell 3 — Find exact file paths
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/datasets/excimusprime/machine-translation-files/eng_embedding_matrix.npy
/kaggle/input/datasets/excimusprime/machine-translation-files/hi_embedding_matrix.npy
/kaggle/input/datasets/excimusprime/machine-translation-files/train_processed.pkl
/kaggle/input/datasets/excimusprime/machine-translation-files/val_processed.pkl
/kaggle/input/datasets/excimusprime/machine-translation-files/eng_vocab.pkl
/kaggle/input/datasets/excimusprime/machine-translation-files/hi_vocab.pkl
/kaggle/input/datasets/excimusprime/machine-translation-files/test_processed.pkl


In [4]:
# Cell 4 — Load all data
DATA_PATH = '/kaggle/input/datasets/excimusprime/machine-translation-files/'

train = pd.read_pickle(DATA_PATH + 'train_processed.pkl')
val   = pd.read_pickle(DATA_PATH + 'val_processed.pkl')
test  = pd.read_pickle(DATA_PATH + 'test_processed.pkl')

with open(DATA_PATH + 'eng_vocab.pkl', 'rb') as f:
    eng_vocab = pickle.load(f)
with open(DATA_PATH + 'hi_vocab.pkl', 'rb') as f:
    hi_vocab = pickle.load(f)

eng_matrix = np.load(DATA_PATH + 'eng_embedding_matrix.npy')
hi_matrix  = np.load(DATA_PATH + 'hi_embedding_matrix.npy')

print(f"Train: {len(train):,}")
print(f"Val:   {len(val):,}")
print(f"Test:  {len(test):,}")
print(f"English vocab: {len(eng_vocab):,} | Matrix: {eng_matrix.shape}")
print(f"Hindi vocab:   {len(hi_vocab):,} | Matrix: {hi_matrix.shape}")

Train: 120,000
Val:   15,000
Test:  15,000
English vocab: 18,671 | Matrix: (18671, 300)
Hindi vocab:   18,715 | Matrix: (18715, 300)


In [5]:
# Cell 5 — Dataset class
class TranslationDataset(Dataset):
    def __init__(self, dataframe):
        self.english = torch.tensor(
            dataframe['eng_padded'].tolist(),
            dtype=torch.long
        )
        self.hindi = torch.tensor(
            dataframe['hi_padded'].tolist(),
            dtype=torch.long
        )

    def __len__(self):
        return len(self.english)

    def __getitem__(self, idx):
        return self.english[idx], self.hindi[idx]

train_dataset = TranslationDataset(train)
val_dataset   = TranslationDataset(val)
test_dataset  = TranslationDataset(test)

print(f"Train dataset: {len(train_dataset):,} pairs")
print(f"Val dataset:   {len(val_dataset):,} pairs")
print(f"Test dataset:  {len(test_dataset):,} pairs")

Train dataset: 120,000 pairs
Val dataset:   15,000 pairs
Test dataset:  15,000 pairs


In [6]:
# Cell 6 — DataLoaders
BATCH_SIZE = 64

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)

print(f"Batch size: {BATCH_SIZE}")
print(f"Training batches per epoch:   {len(train_loader):,}")
print(f"Validation batches per epoch: {len(val_loader):,}")

Batch size: 64
Training batches per epoch:   1,875
Validation batches per epoch: 235


In [7]:
# Cell 7 — BiLSTM Encoder with Self Attention [FIXED]
# Fixes applied:
#   Bug 1: LayerNorm added after tanh projection (stabilises gradient through bottleneck)
#   Bug 2: Residual connection on self-attention (was missing in original)
#   Bug 4: Attention scaling already present (scale = sqrt(hidden_dim*2)) — kept
# NOTE: encoder outputs are kept at hidden_dim*2 so cross-attention in Decoder
#       can use kdim/vdim=hidden_dim*2 (fixes the silent dim-mismatch bug)

class Encoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, dropout, pretrained_embeddings):
        super(Encoder, self).__init__()

        self.hidden_dim = hidden_dim

        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.embedding.weight = nn.Parameter(
            torch.tensor(pretrained_embeddings, dtype=torch.float32)
        )
        self.embedding.weight.requires_grad = True

        # BiLSTM — bidirectional=True reads forward AND backward
        self.bilstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            batch_first=True,
            bidirectional=True
        )

        # Projection: 2*hidden_dim -> hidden_dim for decoder init
        self.fc_hidden = nn.Linear(hidden_dim * 2, hidden_dim)
        self.fc_cell   = nn.Linear(hidden_dim * 2, hidden_dim)

        # FIX Bug 1: LayerNorm after projection to stabilise gradient
        self.norm_hidden = nn.LayerNorm(hidden_dim)
        self.norm_cell   = nn.LayerNorm(hidden_dim)

        # Self-attention operates on 2*hidden_dim encoder outputs
        self.attention_q = nn.Linear(hidden_dim * 2, hidden_dim * 2)
        self.attention_k = nn.Linear(hidden_dim * 2, hidden_dim * 2)
        self.attention_v = nn.Linear(hidden_dim * 2, hidden_dim * 2)

        # LayerNorm on encoder outputs after residual add
        self.norm_enc = nn.LayerNorm(hidden_dim * 2)

        self.dropout = nn.Dropout(dropout)
        self.scale   = math.sqrt(hidden_dim * 2)

    def self_attention(self, hidden_states):
        Q = self.attention_q(hidden_states)
        K = self.attention_k(hidden_states)
        V = self.attention_v(hidden_states)

        scores  = torch.bmm(Q, K.transpose(1, 2)) / self.scale
        weights = torch.softmax(scores, dim=-1)
        weights = self.dropout(weights)
        context = torch.bmm(weights, V)
        return context

    def forward(self, src):
        embedded = self.dropout(self.embedding(src))

        outputs, (hidden, cell) = self.bilstm(embedded)
        # outputs: (batch, seq_len, hidden_dim*2)

        # FIX Bug 2: residual connection on self-attention output + LayerNorm
        attended = self.self_attention(outputs)
        outputs  = self.norm_enc(outputs + attended)
        # outputs stay at hidden_dim*2 — decoder cross-attn uses kdim/vdim

        # Concatenate forward/backward final hidden & cell
        hidden_cat = torch.cat([hidden[0], hidden[1]], dim=1)  # (batch, hidden_dim*2)
        cell_cat   = torch.cat([cell[0],   cell[1]],   dim=1)  # (batch, hidden_dim*2)

        # FIX Bug 1: tanh projection + LayerNorm
        hidden = self.norm_hidden(torch.tanh(self.fc_hidden(hidden_cat))).unsqueeze(0)
        cell   = self.norm_cell(torch.tanh(self.fc_cell(cell_cat))).unsqueeze(0)

        return outputs, hidden, cell

print("BiLSTM Encoder defined — all fixes applied!")


BiLSTM Encoder defined — all fixes applied!


In [8]:
# Cell 8 — Decoder [FIXED v2]
# Fixes applied:
#   Bug 3: cross-attention context concatenated with decoder output before fc_out
#   Bug 4: cross-attention uses kdim=hidden_dim*2, vdim=hidden_dim*2 to match encoder outputs
#           NOTE: MHA output dim = embed_dim = hidden_dim (the query projection dim, not kdim)
#   Bug 5: dropout on combined representation before fc_out
#
# LSTM input_size = embed_dim + hidden_dim  (context dim = hidden_dim, the MHA output dim)
# fc_out input    = hidden_dim + hidden_dim = hidden_dim*2  (lstm_out + context)

class Decoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, dropout, pretrained_embeddings):
        super(Decoder, self).__init__()

        self.hidden_dim = hidden_dim

        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.embedding.weight = nn.Parameter(
            torch.tensor(pretrained_embeddings, dtype=torch.float32)
        )
        self.embedding.weight.requires_grad = True

        # Cross-attention: Q from decoder (hidden_dim), K/V from encoder (hidden_dim*2)
        # MHA output dim = embed_dim = hidden_dim  (the query space dim)
        self.cross_attention = nn.MultiheadAttention(
            embed_dim=hidden_dim,
            num_heads=4,
            dropout=dropout,
            batch_first=True,
            kdim=hidden_dim * 2,
            vdim=hidden_dim * 2
        )

        # LSTM input = embedding (embed_dim) + context (hidden_dim) = embed_dim + hidden_dim
        self.lstm = nn.LSTM(
            input_size=embed_dim + hidden_dim,
            hidden_size=hidden_dim,
            batch_first=True
        )

        # fc_out input = lstm_out (hidden_dim) + context (hidden_dim) = hidden_dim*2
        self.fc_out  = nn.Linear(hidden_dim * 2, vocab_size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, tgt, hidden, cell, encoder_outputs):
        tgt      = tgt.unsqueeze(1)
        embedded = self.dropout(self.embedding(tgt))       # (batch, 1, embed_dim)

        # Cross-attention: Q=decoder hidden, K/V=encoder outputs
        query   = hidden.permute(1, 0, 2)                  # (batch, 1, hidden_dim)
        context, attn_weights = self.cross_attention(
            query,            # Q: (batch, 1, hidden_dim)
            encoder_outputs,  # K: (batch, src_len, hidden_dim*2)
            encoder_outputs   # V: (batch, src_len, hidden_dim*2)
        )
        # context: (batch, 1, hidden_dim)  <-- MHA output dim = embed_dim = hidden_dim

        # LSTM input = [embedding ; context]  => (batch, 1, embed_dim + hidden_dim)
        rnn_input = torch.cat([embedded, context], dim=2)

        output, (hidden, cell) = self.lstm(rnn_input, (hidden, cell))
        # output: (batch, 1, hidden_dim)

        # Concatenate lstm output + context, then project to vocab
        combined   = torch.cat([output.squeeze(1), context.squeeze(1)], dim=1)
        # combined: (batch, hidden_dim + hidden_dim) = (batch, hidden_dim*2)

        prediction = self.fc_out(self.dropout(combined))   # (batch, vocab_size)

        return prediction, hidden, cell, attn_weights

print("Decoder defined — v2 fix applied (LSTM input_size corrected to embed_dim + hidden_dim)!")


Decoder defined — v2 fix applied (LSTM input_size corrected to embed_dim + hidden_dim)!


In [9]:
# Cell 9 — Seq2Seq (unchanged — encoder/decoder fixes propagate automatically)
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super(Seq2Seq, self).__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device  = device

    def forward(self, src, tgt, teacher_forcing_ratio=0.5):
        batch_size     = src.shape[0]
        tgt_len        = tgt.shape[1]
        tgt_vocab_size = self.decoder.fc_out.out_features

        outputs = torch.zeros(batch_size, tgt_len, tgt_vocab_size).to(self.device)

        encoder_outputs, hidden, cell = self.encoder(src)
        decoder_input = tgt[:, 0]

        for t in range(1, tgt_len):
            prediction, hidden, cell, _ = self.decoder(
                decoder_input, hidden, cell, encoder_outputs
            )
            outputs[:, t, :] = prediction

            use_teacher_forcing = torch.rand(1).item() < teacher_forcing_ratio
            if use_teacher_forcing:
                decoder_input = tgt[:, t]
            else:
                decoder_input = prediction.argmax(1)

        return outputs

print("Seq2Seq defined!")


Seq2Seq defined!


In [10]:
# Cell 10 — Initialize model
# hidden_dim=256 kept — fixes make it work correctly now without needing to increase it
EMBED_DIM     = 300
HIDDEN_DIM    = 256
DROPOUT       = 0.3
LEARNING_RATE = 0.001

encoder = Encoder(
    vocab_size=len(eng_vocab),
    embed_dim=EMBED_DIM,
    hidden_dim=HIDDEN_DIM,
    dropout=DROPOUT,
    pretrained_embeddings=eng_matrix
).to(device)

decoder = Decoder(
    vocab_size=len(hi_vocab),
    embed_dim=EMBED_DIM,
    hidden_dim=HIDDEN_DIM,
    dropout=DROPOUT,
    pretrained_embeddings=hi_matrix
).to(device)

model = Seq2Seq(encoder, decoder, device).to(device)

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Model initialized!")
print(f"Trainable parameters: {count_parameters(model):,}")


Model initialized!
Trainable parameters: 24,239,827


In [11]:
# Cell 11 — Loss and optimizer
criterion = nn.CrossEntropyLoss(ignore_index=0)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

print(f"Loss: CrossEntropyLoss (ignoring PAD)")
print(f"Optimizer: Adam (lr={LEARNING_RATE})")

Loss: CrossEntropyLoss (ignoring PAD)
Optimizer: Adam (lr=0.001)


In [12]:
# Cell 12 — Training functions
def train_epoch(model, loader, optimizer, criterion, clip=1.0):
    model.train()
    epoch_loss = 0

    for src, tgt in loader:
        src = src.to(device)
        tgt = tgt.to(device)

        optimizer.zero_grad()
        output = model(src, tgt, teacher_forcing_ratio=0.5)

        output = output[:, 1:, :].reshape(-1, output.shape[-1])
        tgt    = tgt[:, 1:].reshape(-1)

        loss = criterion(output, tgt)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
        optimizer.step()

        epoch_loss += loss.item()

    return epoch_loss / len(loader)


def evaluate(model, loader, criterion):
    model.eval()
    epoch_loss = 0

    with torch.no_grad():
        for src, tgt in loader:
            src = src.to(device)
            tgt = tgt.to(device)

            output = model(src, tgt, teacher_forcing_ratio=0.0)

            output = output[:, 1:, :].reshape(-1, output.shape[-1])
            tgt    = tgt[:, 1:].reshape(-1)

            loss = criterion(output, tgt)
            epoch_loss += loss.item()

    return epoch_loss / len(loader)

print("Training functions defined!")

Training functions defined!


In [13]:
# Cell 13 — Train!
N_EPOCHS      = 15
best_val_loss = float('inf')

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=3
)

for epoch in range(N_EPOCHS):
    train_loss = train_epoch(model, train_loader, optimizer, criterion)
    val_loss   = evaluate(model, val_loader, criterion)

    scheduler.step(val_loss)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), '/kaggle/working/bilstm_best_model.pt')
        print(f"Epoch {epoch+1:02d} -> Train: {train_loss:.4f} | Val: {val_loss:.4f} saved")
    else:
        print(f"Epoch {epoch+1:02d} -> Train: {train_loss:.4f} | Val: {val_loss:.4f}")

print(f"\nTraining complete! Best val loss: {best_val_loss:.4f}")

Epoch 01 -> Train: 5.4785 | Val: 5.4424 saved
Epoch 02 -> Train: 4.6996 | Val: 5.2815 saved
Epoch 03 -> Train: 4.3980 | Val: 5.1895 saved
Epoch 04 -> Train: 4.1962 | Val: 5.1825 saved
Epoch 05 -> Train: 4.0575 | Val: 5.1694 saved
Epoch 06 -> Train: 3.9409 | Val: 5.1512 saved
Epoch 07 -> Train: 3.8581 | Val: 5.1522
Epoch 08 -> Train: 3.7776 | Val: 5.2087
Epoch 09 -> Train: 3.7052 | Val: 5.2706
Epoch 10 -> Train: 3.6470 | Val: 5.2268
Epoch 11 -> Train: 3.4734 | Val: 5.2362
Epoch 12 -> Train: 3.4170 | Val: 5.2380
Epoch 13 -> Train: 3.3788 | Val: 5.2590
Epoch 14 -> Train: 3.3494 | Val: 5.2582
Epoch 15 -> Train: 3.2492 | Val: 5.2976

Training complete! Best val loss: 5.1512


In [14]:
# Cell 14 — Evaluation
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
from sklearn.metrics import f1_score

model.load_state_dict(torch.load('/kaggle/working/bilstm_best_model.pt'))
model.eval()

idx2hi  = {idx: word for word, idx in hi_vocab.items()}
idx2eng = {idx: word for word, idx in eng_vocab.items()}

def indices_to_tokens(indices, idx2word):
    tokens = []
    for idx in indices:
        idx = idx.item()
        if idx == 2:  # EOS
            break
        if idx not in [0, 1]:  # skip PAD and SOS
            tokens.append(idx2word.get(idx, '<UNK>'))
    return tokens

all_predictions = []
all_references  = []
all_pred_flat   = []
all_tgt_flat    = []

print("Evaluating on test set...")
for src, tgt in test_loader:
    src = src.to(device)
    tgt = tgt.to(device)

    with torch.no_grad():
        output = model(src, tgt, teacher_forcing_ratio=0.0)

    pred_indices = output.argmax(dim=2)

    for i in range(pred_indices.shape[0]):
        pred_tokens = indices_to_tokens(pred_indices[i], idx2hi)
        ref_tokens  = indices_to_tokens(tgt[i], idx2hi)

        all_predictions.append(pred_tokens)
        all_references.append([ref_tokens])

        min_len = min(len(pred_tokens), len(ref_tokens))
        if min_len > 0:
            all_pred_flat.extend(pred_tokens[:min_len])
            all_tgt_flat.extend(ref_tokens[:min_len])

print(f"Evaluation done! Total: {len(all_predictions):,}")
print(f"Non-empty predictions: {sum(1 for p in all_predictions if len(p) > 0):,}")

Evaluating on test set...
Evaluation done! Total: 15,000
Non-empty predictions: 15,000


In [15]:
# Cell 15 — Compute Scores
smoothing  = SmoothingFunction().method1
bleu_score = corpus_bleu(all_references, all_predictions,
                          smoothing_function=smoothing)

if all_tgt_flat:
    correct  = sum(p == r for p, r in zip(all_pred_flat, all_tgt_flat))
    accuracy = correct / len(all_tgt_flat)
    unique_labels = list(set(all_tgt_flat))
    f1 = f1_score(all_tgt_flat, all_pred_flat,
                  labels=unique_labels,
                  average='weighted',
                  zero_division=0)
else:
    accuracy = 0
    f1 = 0

print("=" * 50)
print("     BiLSTM MODEL — TEST SET RESULTS")
print("=" * 50)
print(f"BLEU Score:  {bleu_score:.4f}  ({bleu_score*100:.2f}%)")
print(f"Accuracy:    {accuracy:.4f}  ({accuracy*100:.2f}%)")
print(f"F1 Score:    {f1:.4f}  ({f1*100:.2f}%)")
print(f"Val Loss:    {best_val_loss:.4f}")
print("=" * 50)

print("\nFULL COMPARISON TABLE")
print("=" * 50)
print(f"{'Model':<10} | {'BLEU':>8} | {'Accuracy':>10} | {'F1':>8}")
print("-" * 50)
print(f"{'RNN':<10} | {'0.00%':>8} | {'0.00%':>10} | {'N/A':>8}")
print(f"{'LSTM':<10} | {'6.16%':>8} | {'10.76%':>10} | {'8.64%':>8}")
print(f"{'BiLSTM':<10} | {bleu_score*100:>7.2f}% | {accuracy*100:>9.2f}% | {f1*100:>7.2f}%")
print("=" * 50)

     BiLSTM MODEL — TEST SET RESULTS
BLEU Score:  0.0807  (8.07%)
Accuracy:    0.1194  (11.94%)
F1 Score:    0.1035  (10.35%)
Val Loss:    5.1512

FULL COMPARISON TABLE
Model      |     BLEU |   Accuracy |       F1
--------------------------------------------------
RNN        |    0.00% |      0.00% |      N/A
LSTM       |    6.16% |     10.76% |    8.64%
BiLSTM     |    8.07% |     11.94% |   10.35%


In [16]:
# Cell 16 — Sample Translations
print("SAMPLE TRANSLATIONS")
print("=" * 60)

src_batch, tgt_batch = next(iter(test_loader))
src_batch = src_batch.to(device)
tgt_batch = tgt_batch.to(device)

with torch.no_grad():
    output = model(src_batch, tgt_batch, teacher_forcing_ratio=0.0)

pred_indices = output.argmax(dim=2)

for i in range(5):
    src_tokens  = indices_to_tokens(src_batch[i], idx2eng)
    ref_tokens  = indices_to_tokens(tgt_batch[i], idx2hi)
    pred_tokens = indices_to_tokens(pred_indices[i], idx2hi)

    print(f"English:   {' '.join(src_tokens[:15])}")
    print(f"Reference: {' '.join(ref_tokens[:15])}")
    print(f"Predicted: {' '.join(pred_tokens[:15])}")
    print("-" * 60)

SAMPLE TRANSLATIONS
English:   a pakistani court has constituted a <UNK> bench to hear an appeal of jailed former
Reference: पाकिस्तान की एक अदालत ने अल <UNK> भ्रष्टाचार मामले में दोषी <UNK> जाने के खिलाफ
Predicted: पाकिस्तानी मामले में एक मामले में प्रधानमंत्री नवाज शरीफ ने भ्रष्टाचार के मामले में भ्रष्टाचार
------------------------------------------------------------
English:   the explosion caused panic in the area .
Reference: इस विस्फोट के चलते क्षेत्र में हड़कंप मच गया ।
Predicted: विस्फोट में इलाके में दहशत का माहौल हुआ ।
------------------------------------------------------------
English:   government approved this
Reference: सरकार ने दी मंजूरी
Predicted: सरकार ने इस मंजूरी को मंजूरी दी
------------------------------------------------------------
English:   india has a population of 130 crore .
Reference: भारत , मन बना चुका है , भारत के करोड़ लोग मन बना चुके हैं
Predicted: भारत में करोड़ करोड़ की आबादी है ।
------------------------------------------------------------
English: 